In [1]:
import os
import json
import glob
import datetime
import pprint
from math import floor
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

import xarray as xr
from shapely.geometry import box

from pystac_client import Client as psClient
from odc.stac import stac_load, configure_rio

import rioxarray

## Helper functions

In [2]:
def read_json(path):
    with open(path, 'r', encoding="utf-8") as file:
        data = json.load(file)
    return data


def crs_from_bbox(bbox):
    """
    Given a bbox [minx, miny, maxx, maxy],
    return the expected EPSG code (UTM zone).
    """
    # Get centroid
    geom = box(*bbox)
    lon, lat = geom.centroid.x, geom.centroid.y

    # UTM zone calculation
    zone = int(floor((lon + 180) / 6) + 1)

    # EPSG: 326## for northern hemisphere, 327## for southern
    if lat >= 0:
        epsg = 32600 + zone
    else:
        epsg = 32700 + zone

    return f"EPSG:{epsg}"


## Main

In [3]:
root = r'C:\Users\andre\Desktop\tmp\giann'
frames_fp = os.path.join(root, 'frames_1280_overlay_50pc.geojson')

In [4]:
# frames
data = read_json(frames_fp)
frames = data['features']
print(f'Found {len(frames)}...')

Found 2396...


In [5]:
# config
start_date = datetime.datetime(2025, 7, 1)
end_date = datetime.datetime(2025, 8, 1)

catalog = psClient.open("https://data.inpe.br/bdc/stac/v1/")
images_dir = os.path.join(root, "images")
os.makedirs(images_dir, exist_ok=True)

print("Extracting samples...")

for frame in frames:
    id_ = frame["properties"]["id"]
    bbox = frame["bbox"]

    print(f"Processing feature {id_}...", end="")

    print("querying stac...", end="")
    query = catalog.search(
        collections=["S2-16D-2"],
        bbox=bbox,
        datetime=[start_date, end_date],
        query={"eo:cloud_cover": {"lt": 10}},
    )

    items = query.item_collection()

    if len(items) == 0:
        print("no items found, skipping.")
        continue

    print(f"found {len(items)} items...", end="")

    print("loading data...", end="")
    ds = stac_load(
        query.items(),
        bands=["B02", "B03", "B04", "B08", "SCL"],
        resampling={
            "B02": "cubic",
            "B03": "cubic",
            "B04": "cubic",
            "B08": "cubic",
            "SCL": "nearest",
        },
        bbox=bbox,
        groupby="solar_day",
        crs=crs_from_bbox(bbox),
        resolution=10,
    )

    # Mask clouds
    print('masking cloudy pixels...', end='')
    cloud_classes = [1, 2, 3, 8, 9, 10, 11]
    clear_mask = ~ds["SCL"].isin(cloud_classes)

    ds = ds[["B02", "B03", "B04", "B08"]].where(clear_mask)

    # Create a single composite
    print("computing median composite...", end="")
    img = ds.median(dim="time", skipna=True)

    # Set CRS for GeoTIFF
    img = img.rio.write_crs(crs_from_bbox(bbox))

    out_path = os.path.join(images_dir, f"{id_}.tif")

    print(f"saving {os.path.basename(out_path)}...", end="")
    img.rio.to_raster(
        out_path,
        compress="LZW",
        dtype="float32",
    )

    print("done.")

Extracting samples...
Processing feature 1725...querying stac...found 3 items...loading data...masking cloudy pixels...computing median composite...saving 1725.tif...done.
Processing feature 1726...querying stac...found 3 items...loading data...masking cloudy pixels...computing median composite...saving 1726.tif...done.
Processing feature 1727...querying stac...found 3 items...loading data...masking cloudy pixels...computing median composite...saving 1727.tif...done.
Processing feature 1728...querying stac...found 3 items...loading data...masking cloudy pixels...computing median composite...saving 1728.tif...done.
Processing feature 1729...querying stac...found 3 items...loading data...masking cloudy pixels...computing median composite...saving 1729.tif...done.
Processing feature 1730...querying stac...found 3 items...loading data...masking cloudy pixels...computing median composite...saving 1730.tif...done.
Processing feature 1731...querying stac...found 3 items...loading data...maski